# LDOS DAY 2: Making Models

Welcome back!

Yesterday we practiced exploratory data analysis (EDA) as a means of analyzing, interpreting, and measuring data on a synthetic load-balancing dataset (hence the p < 0.00000 values you may have seen). Today, we're switching to the **real thing**, training a baseline **Perceptron** on a sample of LDOS' RAID 1 dataset.

## Respond Here: Double-click on this cell and add your names below.

Team Member Names:

Group #:

Team Name:

# Loading Libraries

In this notebook, we’ll use:

- *polars* for working with dataframes
- *pandas* for working with tables of data
- *matplotlib* and *seaborn* for creating charts and graphs
- *sklearn* for our models
- *numpy* for data normalization

We'll load them in the next code cell. Click on the play button next to the cell to run just the cell below.

In [ ]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sklearn

# Background

As a reminder, we're working with a simple replication of the **R**edundant **A**rray of **I**nexpensive **D**isks data storage configuration that uses disk configuration to provide fault tolerance and performant reads at the cost of storage and write performance.

# Data Exploration

Just like yesterday, before jumping straight into building our model, we need to understand out data. Lets start with just one file.

In [ ]:
df_read = pl.read_parquet('1.parquet')
print(df_read.columns)

# Respond Here: What do these columns mean?

*Hint: We saw most of these in the slides, but try and fill in a one-line description for each in your own words.*

- `cpu`:
- `device`:
- `sector`:
- `block_io_bytes`:
- `queue_length_segment_ios`:
- `queue_length_4k_ios`:
- `block_io_flags_string`:
- `block_latency_us`:
- `block_io_latency_us`:
- `measured_latency_us`:
- `label90`:

# Visual Analysis

Now, lets look at the shape and basic stats

In [ ]:
# How many rows and columns?
print(df_read.shape)

# Convert to pandas just for summary tools
df_pd = df_read.to_pandas()
df_pd.describe()

# Respond Here: First Impressions

1. How many rows are in this single file?

2. Which features don't necessarily benefit from these statistics?

3. Which latency column has the largest maximum value? Are they about the same? Does that surprise you?

# Visualizing the Latency Columns

Remember the hint from the slides? If we plot the distribution of latencies, we can often find a natural "fast vs. slow" cutoff.

In [ ]:
# Define figure dimensions
plt.figure(figsize=(8, 5))

# Zoom into the window near our point of inflection
zoom_window = df_pd['block_io_latency_us'].between(0, 800)

# Histogram split into 100 bins
df_pd.loc[zoom_window, 'block_io_latency_us'].hist(bins=100, edgecolor='black')

# Set title and axis labels
plt.title('Partial distribution of block_io_latency_us')
plt.xlabel('Latency (microseconds)')
plt.ylabel('Frequency')

# Force x-axis to only show 0-800
plt.xlim(0, 800)

# Display the chart
plt.show()

# Respond Here: Analyzing Latency

1. Where does most of the data fall relative to the line at 200μs?

2. Does this histogram have one main grouping, or multiple clusters? What might cause that?

# Dropping Features

While having latency as a feature is immensely helpful when analyzing our data, at the moment we need to make a prediction, columns like latency wouldn't be known yet!

The whole point of our model is to help the OS decide how to classify an operation *before* I/O finished. Anything thats only known *after* the operation completes **cannot** be used as an input feature. Using such data is whats called **data leakage** and will almost certainly invaldate your findings.

Lets drop these columns, keeping only `label90` as our target.

In [ ]:
# Drop columns that represent future information or unused target labels
df_drop = df_read.drop(['collection_id', 'block_io_latency_us', 'block_latency_us', 'measured_latency_us', 'label85', 'label95', 'finish_ts_uptime_us'])

# Display the remaining columns
display(df_drop.columns)

# Respond Here: Feature Selection

1. What would happen if we accidenly trained using `block_io_latency_us` as a feature instead of dropping it? Would we even need a model?

# Quantizing `block_io_flags_string`

`block_io_flags_string` is a text column representing the type of operation as a string. Since models can't use text directly, we need numeric features instead. Booleans solve the labeling issue.

Quantizing means converting continuous or categorical things into discrete buckets. Here, we split each flag string into a separate True/False column, assigning a boolean for each possible flag (`Read`, `Write`, `Sync`, `Idle`, `NoMerge`).

In [ ]:
# Import our custom helper function
from library import process_block_io_flags

# Seperate flags into discrete columns
df = process_block_io_flags(df_drop)

# Display the new columns
display(df.columns)

# Respond Here: Library Analysis

Go ahead and open `library.py` and read through the method we call, `process_block_io_flags`.

1. How does it figure out what the possible flags are?

2. What does `.str.contains(flag)` do for each new column?


# Feature Selection Continued

Since we're trying to improve RAID 1's read-scheduling policy, we want to filter our data to be exclusively reads.

**We can only make predictions about reads**

In [ ]:
# Only keep 'Read' operations
df_reads = df.filter(pl.col('Read'))

# Print how many rows remain
print("Number of read operations:", df_reads.shape[0])

# Display the first few rows
display(df_reads.head())

# Respond Here: Feature Selection

1. Approximately what fraction of the original rows were reads?

2. Does this match what we might expect from a system implementing RAID 1? Why or why not?

# Test/Train Splits

Now, we split our data into a **training set** we'll use to fit the model and a **test set** we'll use to check the model's ability to generalize to unseen data.

In [ ]:
# Add a row index column to the DataFrame
df_reads_indexed = df_reads.with_row_index()

# Split the data into training and testing sets (80-20 split)
train_df = df_reads_indexed.sample(fraction=0.8, with_replacement=False, seed=42)
test_df = df_reads_indexed.join(train_df.select('index'), on='index', how='anti')

# Drop the index column from the split dataframes
train_df = train_df.drop('index')
test_df = test_df.drop('index')

# Confirm our split
print("Training data shape:", train_df.shape)
print("Testing data shape:", test_df.shape)

# Respond Here: Test/Train Splits

1. What might go wrong if we train on 100% of our data and tested on that same data?

2. Why is `seed=42` important here? You may have to look this up.

# Splitting Features and Target

Now, lets seperate our inputs from our target (`label90`, what we're trying to predict), and convert to NumPy arrays, the input format scikit-learn expects.

# Data Normalization

Models often perform better with data in the range -1 to 1. Lets normalize our data to 0 to 1 (which is close enough for now). There are various ways to do this, but for now we will just divide each value by the maximum of it's respective column (plus a small amount to avoid dividing by zero).

In [ ]:
# Seperate our training features and target
train_features = train_df.drop('label90')
train_target = train_df['label90']

# Seperate our test features and target
test_features = test_df.drop('label90')
test_target = test_df['label90']

# Convert training sets into NumPy arrays
train_features_np = train_features.to_numpy()
train_target_np = train_target.to_numpy()

# Convert test sets into NumPy arrays
test_features_np = test_features.to_numpy()
test_target_np = test_target.to_numpy()

# Normalize our data
norm_np = np.abs(train_features_np).max(axis=0) + 0.0001
train_features_np = train_features_np / norm_np
test_features_np = test_features_np / norm_np

# View the shape of our features
print("Feature columns:", train_features.columns)
print("Training features shape:", train_features_np.shape)
print("Testing features shape:", test_features_np.shape)

# Baseline Model: Perceptrons

Y'all learned about Perceptrons earlier, lets see what they can do!

We're starting with a simple MLPClassifier with **no hidden layers**. This is mathematically equivalent to **logistic regression**, a single layer weighing each feature and producing a fast/slow prediction.

In [ ]:
from sklearn.neural_network import MLPClassifier

# Define the MLPClassifier model with no hidden layers
baseline_percept = MLPClassifier(hidden_layer_sizes=(), activation='relu', solver='adam', max_iter=500, random_state=42)

# Print the model configuration
print(baseline_percept)

Now, lets use the split training set we prepared earlier to train out model.

In [ ]:
# Train the scikit-learn model
print("Training the Baseline Perceptron (MLPClassifier)...")
baseline_percept.fit(train_features_np, train_target_np)
print("Training complete.")

# Model Evaluation

Lets see how well that did. We'll compute **accuracy**, precision, recall, and **F1 score**, plus a **confusion matrix**.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Make predictions on the test set
test_predictions = baseline_percept.predict(test_features_np)

# Calculate evaluation metrics
accuracy = accuracy_score(test_target_np, test_predictions)
precision = precision_score(test_target_np, test_predictions)
recall = recall_score(test_target_np, test_predictions)
f1 = f1_score(test_target_np, test_predictions)
conf_matrix = confusion_matrix(test_target_np, test_predictions)

# Print the results
print(f"Accuracy on the test set: {accuracy:.4f}")
print(f"Precision on the test set: {precision:.4f}")
print(f"Recall on the test set: {recall:.4f}")
print(f"F1-Score on the test set: {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)

# Respond Here: Reading the Confusion Matrix

The confusion matrix is laid out like this:

```
                 Predicted Slow   Predicted Fast
Actual Slow     [   TN              FP        ]
Actual Fast     [   FN              TP        ]
```

1. Is your precision higher or lower than your recall? What does that tell you about the kinds of mistakes your model might be making?

2. If `label90` is mostly `True` (fast) in this dataset, what accuracy might a model get by always predicting "fast" without really learning anything? Compared to your model's accuracy, do you think this model learned something useful?

# Model Speed

Remember: model performance isn't just accuracy. A model like the predictive model we're running might run up to 3 times for a single read, with billions of read operations occuring every second in a modern OS. Lets measure how long a single prediction takes.

In [ ]:
# Fetch our helper function
from library import time_model_execution

# Warm up our baseline perceptron with 10,000 predictions, then time one
ts_ns = time_model_execution(baseline_percept, test_features_np)

# Print the result in ns and us
print(f"Time to execute a single prediction: {ts_ns:.2f}ns ({ts_ns/1000:.2f}us)")

# Respond Here: Model Timing

1. If a single read could trigger this model **3 times** (once per disk), what's the worst-case latency?

2. How does that latency compare to the fast/slow threshold we defined earlier? What might that tell you about why model *size and speed* model just as much as accuracy?

# Hyperparameter Tuning: Hidden Layers

Our current model has **zero hidden layers**, making it pure logistic regression. By adding hidden layers, we let the model learn more complex, non-linear patterns, potentially at the cost of speed.

Create a second model called `second_percept`. By changing `hidden_layer_sizes`, you'll add a number of hidden layers to your model. Lets start by adding *one* hidden layer with a number of neurons of your choice (try between 1 and 10).

Hint: `hidden_layer_sizes` is a parameter that takes in a *tuple*, which you can think of as a list if you're unfamiliar. Each element in the tuple represents a number of neurons in that specific layer, while the length of the tuple represents the number of hidden layers you're adding (so (5,) represents a single layer with 5 neurons, and (4, 4) represents two layers with 4 neurons each).

In [ ]:
# CODE HERE: pick a number of hidden layers and neurons
second_percept = MLPClassifier(hidden_layer_sizes=(), activation='relu', solver='adam', max_iter=500, random_state=42)
print(second_percept, end="\n\n")

# Train
print("Training the Second Perceptron (MLPClassifier)...")
second_percept.fit(train_features_np, train_target_np)
print("Training complete.\n")

# Evaluate
test_predictions = second_percept.predict(test_features_np)
accuracy = accuracy_score(test_target_np, test_predictions)
precision = precision_score(test_target_np, test_predictions)
recall = recall_score(test_target_np, test_predictions)
f1 = f1_score(test_target_np, test_predictions)
conf_matrix = confusion_matrix(test_target_np, test_predictions)

# Time it
ts_ns = time_model_execution(second_percept, test_features_np)

# Print the results
print(f"Accuracy on the test set: {accuracy:.4f}")
print(f"Precision on the test set: {precision:.4f}")
print(f"Recall on the test set: {recall:.4f}")
print(f"F1-Score on the test set: {f1:.4f}")
print(f"Time to execute: {ts_ns:.2f}ns ({ts_ns/1000:.2f}us)")
print("\nConfusion Matrix:")
print(conf_matrix)

# Respond Here: Compare percept_baseline vs. second_percept

baseline_percept

- accuracy:
- f1 score:
- speed:

second_percept

- accuracy:
- f1 score:
- speed:

1. Did adding a hidden layer improve accuracy? By how much?

2. Did adding a hidden layer change speed? Does this make sense given what a hidden layer adds computationally?

3. Try at least one more value for `hidden_layer_sizes`, either changine the number of neurons or hidden layers. What pattern do you notice between model size, accuracy, and speed?

## Write down your results

Before you go, write down your `baseline_percept` and best `second_percept` results, you'll need them later

## Download and Turn in this notebook

Make sure you've run all the code cells and answered every "Respond Here" cell, then, download this notebook as "Group_[group number]_Module_2.ipynb" and submit at this link: https://forms.gle/dX2j6PgT2SsPjZFWA